# LIBERO eval — **노드 B** (GPU 4장)

노드 B 가 **학습한 것과 같은 (모델,seed)** 를 평가한다(공유 FS 아니어도 안전).

- **150k 체크포인트 × 500ep**, 외란 없음(표준 `lerobot_eval`). action(.pt) 기록 → jerk/LDJ/SPARC/SignFlip 측정 가능.
- ⚠️ **LIBERO 시뮬 필요**(robosuite/libero 설치). 동시 env = batch × 10(task) 이라 batch 를 작게(=`LIBERO_EVAL_BATCH`).
- 결과 = `eval_clean/libero_10/<model>/seed<N>/rep0/` → `reports/09_report_sr`·`utils/0l_collect_libero` 가 자동 수집.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

In [ ]:
NODE = 'B'
N_EP = 500                              # 은지 요청 = 500ep
pairs = [(t, s) for t, s, _ in cf.node_jobs(NODE)]   # 이 노드가 학습한 (모델,seed)
gpus = cf.available_gpus()[:cf.NODE_GPUS[NODE]]
print(f'노드 {NODE} | GPU {gpus} | {len(pairs)}쌍 × {N_EP}ep | task=libero_10')
for t, s in pairs:
    print(f'   {t:10} seed{s}')

## 1) 사전 체크 — 150k 체크포인트가 있나
`⚠`/`❌` 뜨면 그 (모델,seed)는 아직 150k 미도달(학습 먼저).


In [ ]:
for t, s in pairs:
    got = cf.resolved_ckpt_step(t, s, step=cf.CKPT_STEP)
    flag = 'OK' if got == cf.CKPT_STEP else ('❌ 없음' if got is None else f'⚠ 최근접 {got:,}')
    print(f'   {t:10} seed{s}: {flag}')

## 2) 실행 — 500ep, GPU 하나당 eval 하나(OOM 방지)


In [ ]:
cf.run_libero_eval_jobs(pairs, gpus=gpus, n_episodes=N_EP)

## 3) 결과 (SR + 저장 영상 수)


In [ ]:
import glob
print(f"{'MODEL':<12}{'SEED':>5}{'SR':>9}{'VIDEOS':>8}")
print('-' * 34)
for t, s in pairs:
    st = cf.get_eval_status(t, s, 'libero_10')
    sr = f"{st['sr']*100:.1f}%" if st['sr'] is not None else '-'
    vids = glob.glob(str(cf.eval_clean_dir(t, s, 'libero_10') / '**' / '*.mp4'), recursive=True)
    print(f'{t:<12}{s:>5}{sr:>9}{len(vids):>8}')